# Phase 4.1 — Portfolio Optimizer Example

Side-by-side: mean-variance vs risk-parity on a synthetic universe.


In [ ]:
from unittest.mock import MagicMock

import numpy as np
import pandas as pd

from ah_research.portfolio.constructor import Constraint
from ah_research.portfolio.optimizer import mean_variance, risk_parity

In [ ]:
# Synthetic universe with real ticker format (required by WeightsSchema)
symbols = [
    "600519.SH",
    "000858.SZ",
    "601398.SH",
    "600036.SH",
    "000333.SZ",
    "600276.SH",
    "600887.SH",
    "601318.SH",
]
rng = np.random.default_rng(0)
dates = pd.bdate_range("2025-01-01", periods=260)
rows = []
for sym in symbols:
    r = rng.normal(0, 0.012, size=len(dates))
    for d, ret in zip(dates, r, strict=True):
        rows.append({"ds": d, "symbol": sym, "close_hfq": 100.0, "total_return": ret})
prices = pd.DataFrame(rows)
repo = MagicMock()
repo.get_prices.return_value = prices

## MV with historical-mean mu

In [ ]:
res_mv = mean_variance(
    symbols,
    pd.Timestamp("2025-12-31"),
    repo,
    constraints=[Constraint.max_weight(0.25)],
    risk_aversion=5.0,
)
print(res_mv.to_markdown())

## Risk-parity

In [ ]:
res_rp = risk_parity(symbols, pd.Timestamp("2025-12-31"), repo)
print(res_rp.to_markdown())

## Side-by-side weights

In [ ]:
compare = pd.DataFrame({"mv": res_mv.weights, "risk_parity": res_rp.weights}).round(4)
compare

In [ ]:
assert abs(res_mv.weights.sum() - 1.0) < 1e-6
assert abs(res_rp.weights.sum() - 1.0) < 1e-6
assert res_rp.risk_contributions is not None
print("All checks passed")